In [1]:
try:
    spark.stop()
except Exception as e:
    print("Spark was already stopped or unavailable:", e)

Spark was already stopped or unavailable: name 'spark' is not defined


In [2]:
#Spark initialization cell
# tells spark to run locally on mac, use all CPU cores, call this investigation "HealthcareInvestigation"
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("HealthcareInvestigation")
    .master("local[2]")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .config("spark.ui.enabled", "false")
    .config("spark.driver.memory", "2g")
    .config("spark.executor.memory", "2g")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

print("Spark version:", spark.version)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/05 08:31:50 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 4.0.4


In [3]:
#read beneficiary data
#tells spark to go to data/raw/beneficiary, read this csv, first row as col type, inferdata type, create a dataframe (distributed table spark can work with)
beneficiary_path = "../data/raw/beneficiary/DE1_0_2008_Beneficiary_Summary_File_Sample_2.csv"

beneficiary_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(beneficiary_path)
)

beneficiary_df.printSchema()

root
 |-- DESYNPUF_ID: string (nullable = true)
 |-- BENE_BIRTH_DT: integer (nullable = true)
 |-- BENE_DEATH_DT: integer (nullable = true)
 |-- BENE_SEX_IDENT_CD: integer (nullable = true)
 |-- BENE_RACE_CD: integer (nullable = true)
 |-- BENE_ESRD_IND: string (nullable = true)
 |-- SP_STATE_CODE: integer (nullable = true)
 |-- BENE_COUNTY_CD: integer (nullable = true)
 |-- BENE_HI_CVRAGE_TOT_MONS: integer (nullable = true)
 |-- BENE_SMI_CVRAGE_TOT_MONS: integer (nullable = true)
 |-- BENE_HMO_CVRAGE_TOT_MONS: integer (nullable = true)
 |-- PLAN_CVRG_MOS_NUM: integer (nullable = true)
 |-- SP_ALZHDMTA: integer (nullable = true)
 |-- SP_CHF: integer (nullable = true)
 |-- SP_CHRNKIDN: integer (nullable = true)
 |-- SP_CNCR: integer (nullable = true)
 |-- SP_COPD: integer (nullable = true)
 |-- SP_DEPRESSN: integer (nullable = true)
 |-- SP_DIABETES: integer (nullable = true)
 |-- SP_ISCHMCHT: integer (nullable = true)
 |-- SP_OSTEOPRS: integer (nullable = true)
 |-- SP_RA_OA: integer (

In [4]:
# show first 5 beneficiaries
#DESYNPUF_ID → beneficiary identifier
beneficiary_df.show(5, truncate=False)

+----------------+-------------+-------------+-----------------+------------+-------------+-------------+--------------+-----------------------+------------------------+------------------------+-----------------+-----------+------+-----------+-------+-------+-----------+-----------+-----------+-----------+--------+-----------+-----------+---------+---------+-----------+---------+---------+------------+----------+----------+
|DESYNPUF_ID     |BENE_BIRTH_DT|BENE_DEATH_DT|BENE_SEX_IDENT_CD|BENE_RACE_CD|BENE_ESRD_IND|SP_STATE_CODE|BENE_COUNTY_CD|BENE_HI_CVRAGE_TOT_MONS|BENE_SMI_CVRAGE_TOT_MONS|BENE_HMO_CVRAGE_TOT_MONS|PLAN_CVRG_MOS_NUM|SP_ALZHDMTA|SP_CHF|SP_CHRNKIDN|SP_CNCR|SP_COPD|SP_DEPRESSN|SP_DIABETES|SP_ISCHMCHT|SP_OSTEOPRS|SP_RA_OA|SP_STRKETIA|MEDREIMB_IP|BENRES_IP|PPPYMT_IP|MEDREIMB_OP|BENRES_OP|PPPYMT_OP|MEDREIMB_CAR|BENRES_CAR|PPPYMT_CAR|
+----------------+-------------+-------------+-----------------+------------+-------------+-------------+--------------+-----------------------+

26/10/05 08:31:53 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


In [5]:
#sanity check: number or rows and unique beneficiaries
print("Total rows:", beneficiary_df.count())
print("Unique beneficiaries:", beneficiary_df.select("DESYNPUF_ID").distinct().count())

Total rows: 116395
Unique beneficiaries: 116395


In [6]:
#shows cols for the first 10 beneficiaries
beneficiary_df.select(
    "DESYNPUF_ID",
    "BENE_BIRTH_DT",
    "BENE_SEX_IDENT_CD",
    "SP_DIABETES",
    "SP_CHF",
    "MEDREIMB_CAR"
).show(10)

+----------------+-------------+-----------------+-----------+------+------------+
|     DESYNPUF_ID|BENE_BIRTH_DT|BENE_SEX_IDENT_CD|SP_DIABETES|SP_CHF|MEDREIMB_CAR|
+----------------+-------------+-----------------+-----------+------+------------+
|00000B48BCF4AD29|     19230901|                2|          1|     1|      6260.0|
|0000525AB30E4DEF|     19201001|                2|          1|     1|      3830.0|
|00009C897C3D8372|     19320101|                1|          1|     1|      1540.0|
|0001168CE43BE51B|     19340901|                2|          2|     2|         0.0|
|0002E494BC87CE10|     19140701|                1|          2|     2|         0.0|
|0002EC0BCA99CACF|     19421201|                1|          2|     2|         0.0|
|0002FABBFD6D1C5C|     19240701|                2|          1|     1|      1140.0|
|000364233C3E3682|     19351201|                1|          2|     1|      1410.0|
|000433285C8D2349|     19270901|                2|          2|     2|      1740.0|
|000

In [7]:
carrier_path = "../data/raw/carrier/DE1_0_2008_to_2010_Carrier_Claims_Sample_2A.csv"

carrier_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(carrier_path)
)

carrier_df.printSchema()

root
 |-- DESYNPUF_ID: string (nullable = true)
 |-- CLM_ID: long (nullable = true)
 |-- CLM_FROM_DT: integer (nullable = true)
 |-- CLM_THRU_DT: integer (nullable = true)
 |-- ICD9_DGNS_CD_1: string (nullable = true)
 |-- ICD9_DGNS_CD_2: string (nullable = true)
 |-- ICD9_DGNS_CD_3: string (nullable = true)
 |-- ICD9_DGNS_CD_4: string (nullable = true)
 |-- ICD9_DGNS_CD_5: string (nullable = true)
 |-- ICD9_DGNS_CD_6: string (nullable = true)
 |-- ICD9_DGNS_CD_7: string (nullable = true)
 |-- ICD9_DGNS_CD_8: string (nullable = true)
 |-- PRF_PHYSN_NPI_1: long (nullable = true)
 |-- PRF_PHYSN_NPI_2: long (nullable = true)
 |-- PRF_PHYSN_NPI_3: long (nullable = true)
 |-- PRF_PHYSN_NPI_4: long (nullable = true)
 |-- PRF_PHYSN_NPI_5: long (nullable = true)
 |-- PRF_PHYSN_NPI_6: long (nullable = true)
 |-- PRF_PHYSN_NPI_7: long (nullable = true)
 |-- PRF_PHYSN_NPI_8: long (nullable = true)
 |-- PRF_PHYSN_NPI_9: long (nullable = true)
 |-- PRF_PHYSN_NPI_10: long (nullable = true)
 |-- PRF_

In [8]:
carrier_df.printSchema()

root
 |-- DESYNPUF_ID: string (nullable = true)
 |-- CLM_ID: long (nullable = true)
 |-- CLM_FROM_DT: integer (nullable = true)
 |-- CLM_THRU_DT: integer (nullable = true)
 |-- ICD9_DGNS_CD_1: string (nullable = true)
 |-- ICD9_DGNS_CD_2: string (nullable = true)
 |-- ICD9_DGNS_CD_3: string (nullable = true)
 |-- ICD9_DGNS_CD_4: string (nullable = true)
 |-- ICD9_DGNS_CD_5: string (nullable = true)
 |-- ICD9_DGNS_CD_6: string (nullable = true)
 |-- ICD9_DGNS_CD_7: string (nullable = true)
 |-- ICD9_DGNS_CD_8: string (nullable = true)
 |-- PRF_PHYSN_NPI_1: long (nullable = true)
 |-- PRF_PHYSN_NPI_2: long (nullable = true)
 |-- PRF_PHYSN_NPI_3: long (nullable = true)
 |-- PRF_PHYSN_NPI_4: long (nullable = true)
 |-- PRF_PHYSN_NPI_5: long (nullable = true)
 |-- PRF_PHYSN_NPI_6: long (nullable = true)
 |-- PRF_PHYSN_NPI_7: long (nullable = true)
 |-- PRF_PHYSN_NPI_8: long (nullable = true)
 |-- PRF_PHYSN_NPI_9: long (nullable = true)
 |-- PRF_PHYSN_NPI_10: long (nullable = true)
 |-- PRF_

In [9]:
carrier_df.limit(5).show(truncate=False)

+----------------+---------------+-----------+-----------+--------------+--------------+--------------+--------------+--------------+--------------+--------------+--------------+---------------+---------------+---------------+---------------+---------------+---------------+---------------+---------------+---------------+----------------+----------------+----------------+----------------+---------+---------+---------+---------+---------+---------+---------+---------+---------+----------+----------+----------+----------+----------+----------+----------+----------+----------+----------+----------+----------+----------+-----------+-----------+-----------+-----------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+-------------------+-------------------+-------------------+-------------------+--------------------------+--------------------------+------------------------

In [10]:
from pyspark.sql import functions as F

claim_lines_1 = (
    carrier_df
    .select(
        "DESYNPUF_ID",
        "CLM_ID",
        "CLM_FROM_DT",
        "CLM_THRU_DT",
        F.col("PRF_PHYSN_NPI_1").alias("provider_npi"),
        F.col("HCPCS_CD_1").alias("hcpcs_code"),
        F.col("LINE_NCH_PMT_AMT_1").alias("payment_amount"),
        F.col("LINE_ALOWD_CHRG_AMT_1").alias("allowed_charge"),
        F.col("LINE_BENE_PTB_DDCTBL_AMT_1").alias("beneficiary_deductible"),
        F.col("LINE_COINSRNC_AMT_1").alias("coinsurance_amount"),
        F.col("LINE_BENE_PRMRY_PYR_PD_AMT_1").alias("primary_payer_amount"),
        F.col("LINE_ICD9_DGNS_CD_1").alias("diagnosis_code")
    )
)

claim_lines_1.show(10, truncate=False)

+----------------+---------------+-----------+-----------+------------+----------+--------------+--------------+----------------------+------------------+--------------------+--------------+
|DESYNPUF_ID     |CLM_ID         |CLM_FROM_DT|CLM_THRU_DT|provider_npi|hcpcs_code|payment_amount|allowed_charge|beneficiary_deductible|coinsurance_amount|primary_payer_amount|diagnosis_code|
+----------------+---------------+-----------+-----------+------------+----------+--------------+--------------+----------------------+------------------+--------------------+--------------+
|00000B48BCF4AD29|737133358637027|20080216   |20080216   |674382001   |99213     |70.0          |110.0         |0.0                   |20.0              |0.0                 |7020          |
|00000B48BCF4AD29|737783362300937|20080219   |20080219   |9798189321  |73550     |280.0         |390.0         |0.0                   |80.0              |0.0                 |9597          |
|00000B48BCF4AD29|737513360019665|20080220   

In [11]:
#key normalization step, each procedure gets it's own row
from functools import reduce

claim_line_dfs = []

for i in range(1, 14):
    claim_line = carrier_df.select(
        "DESYNPUF_ID",
        "CLM_ID",
        "CLM_FROM_DT",
        "CLM_THRU_DT",
        F.col(f"PRF_PHYSN_NPI_{i}").alias("provider_npi"),
        F.col(f"HCPCS_CD_{i}").alias("hcpcs_code"),
        F.col(f"LINE_NCH_PMT_AMT_{i}").alias("payment_amount"),
        F.col(f"LINE_ALOWD_CHRG_AMT_{i}").alias("allowed_charge"),
        F.col(f"LINE_BENE_PTB_DDCTBL_AMT_{i}").alias("beneficiary_deductible"),
        F.col(f"LINE_COINSRNC_AMT_{i}").alias("coinsurance_amount"),
        F.col(f"LINE_BENE_PRMRY_PYR_PD_AMT_{i}").alias("primary_payer_amount"),
        F.col(f"LINE_ICD9_DGNS_CD_{i}").alias("diagnosis_code")
    )

    claim_line_dfs.append(claim_line)

claim_lines = reduce(
    lambda df1, df2: df1.unionByName(df2),
    claim_line_dfs
)

print("Normalized claim-line table:")
claim_lines.show(10, truncate=False)

Normalized claim-line table:
+----------------+---------------+-----------+-----------+------------+----------+--------------+--------------+----------------------+------------------+--------------------+--------------+
|DESYNPUF_ID     |CLM_ID         |CLM_FROM_DT|CLM_THRU_DT|provider_npi|hcpcs_code|payment_amount|allowed_charge|beneficiary_deductible|coinsurance_amount|primary_payer_amount|diagnosis_code|
+----------------+---------------+-----------+-----------+------------+----------+--------------+--------------+----------------------+------------------+--------------------+--------------+
|00000B48BCF4AD29|737133358637027|20080216   |20080216   |674382001   |99213     |70.0          |110.0         |0.0                   |20.0              |0.0                 |7020          |
|00000B48BCF4AD29|737783362300937|20080219   |20080219   |9798189321  |73550     |280.0         |390.0         |0.0                   |80.0              |0.0                 |9597          |
|00000B48BCF4AD2

In [12]:
#Filter empty claims, see how much table has shrunk
claim_lines = (
    claim_lines
    .filter(F.col("hcpcs_code").isNotNull())
)

claim_lines.show(10, truncate=False)

print("Total claim lines:", claim_lines.count())

+----------------+---------------+-----------+-----------+------------+----------+--------------+--------------+----------------------+------------------+--------------------+--------------+
|DESYNPUF_ID     |CLM_ID         |CLM_FROM_DT|CLM_THRU_DT|provider_npi|hcpcs_code|payment_amount|allowed_charge|beneficiary_deductible|coinsurance_amount|primary_payer_amount|diagnosis_code|
+----------------+---------------+-----------+-----------+------------+----------+--------------+--------------+----------------------+------------------+--------------------+--------------+
|00000B48BCF4AD29|737133358637027|20080216   |20080216   |674382001   |99213     |70.0          |110.0         |0.0                   |20.0              |0.0                 |7020          |
|00000B48BCF4AD29|737783362300937|20080219   |20080219   |9798189321  |73550     |280.0         |390.0         |0.0                   |80.0              |0.0                 |9597          |
|00000B48BCF4AD29|737513360019665|20080220   

Total claim lines: 4425332


In [13]:
#saving normalized dataset to parquet
claim_lines.write.mode("overwrite").parquet(
    "../data/processed/claim_lines"
)

In [14]:
import os

print(os.path.exists("../data/processed/claim_lines"))
print(os.listdir("../data/processed/claim_lines")[:5])

True
['part-00127-0388a17f-f27d-4036-bd4c-bec2a0fa8e7a-c000.snappy.parquet', '.part-00087-0388a17f-f27d-4036-bd4c-bec2a0fa8e7a-c000.snappy.parquet.crc', 'part-00026-0388a17f-f27d-4036-bd4c-bec2a0fa8e7a-c000.snappy.parquet', '.part-00022-0388a17f-f27d-4036-bd4c-bec2a0fa8e7a-c000.snappy.parquet.crc', 'part-00052-0388a17f-f27d-4036-bd4c-bec2a0fa8e7a-c000.snappy.parquet']


In [15]:
#verify spark can read dataset back in
claim_lines_parquet = spark.read.parquet(
    "../data/processed/claim_lines"
)

claim_lines_parquet.printSchema()

claim_lines_parquet.show(5, truncate=False)

root
 |-- DESYNPUF_ID: string (nullable = true)
 |-- CLM_ID: long (nullable = true)
 |-- CLM_FROM_DT: integer (nullable = true)
 |-- CLM_THRU_DT: integer (nullable = true)
 |-- provider_npi: long (nullable = true)
 |-- hcpcs_code: string (nullable = true)
 |-- payment_amount: double (nullable = true)
 |-- allowed_charge: double (nullable = true)
 |-- beneficiary_deductible: double (nullable = true)
 |-- coinsurance_amount: double (nullable = true)
 |-- primary_payer_amount: double (nullable = true)
 |-- diagnosis_code: string (nullable = true)

+----------------+---------------+-----------+-----------+------------+----------+--------------+--------------+----------------------+------------------+--------------------+--------------+
|DESYNPUF_ID     |CLM_ID         |CLM_FROM_DT|CLM_THRU_DT|provider_npi|hcpcs_code|payment_amount|allowed_charge|beneficiary_deductible|coinsurance_amount|primary_payer_amount|diagnosis_code|
+----------------+---------------+-----------+-----------+---------

In [16]:
#build provider level features, turn millions of individual claims into smaller tables were one row = on provider
provider_features = (
    claim_lines_parquet
    .groupBy("provider_npi")
    .agg(
        F.count("*").alias("total_claim_lines"),
        F.countDistinct("CLM_ID").alias("unique_claims"),
        F.countDistinct("DESYNPUF_ID").alias("unique_beneficiaries"),
        F.sum("payment_amount").alias("total_payment"),
        F.avg("payment_amount").alias("avg_payment"),
        F.sum("allowed_charge").alias("total_allowed_charge"),
        F.avg("allowed_charge").alias("avg_allowed_charge"),
        F.countDistinct("hcpcs_code").alias("unique_hcpcs")
    )
)

provider_features.show(10, truncate=False)

+------------+-----------------+-------------+--------------------+-------------+------------------+--------------------+------------------+------------+
|provider_npi|total_claim_lines|unique_claims|unique_beneficiaries|total_payment|avg_payment       |total_allowed_charge|avg_allowed_charge|unique_hcpcs|
+------------+-----------------+-------------+--------------------+-------------+------------------+--------------------+------------------+------------+
|27409610    |32               |22           |20                  |1970.0       |61.5625           |2630.0              |82.1875           |25          |
|78794905    |6                |4            |4                   |340.0        |56.666666666666664|410.0               |68.33333333333333 |6           |
|114630046   |5                |3            |3                   |80.0         |16.0              |70.0                |14.0              |5           |
|231233384   |25               |10           |10                  |1200.0   

In [17]:
#monthly provider activity
from pyspark.sql import functions as F

claim_lines_with_month = (
    claim_lines_parquet
    .withColumn(
        "claim_date",
        F.to_date(
            F.col("CLM_FROM_DT").cast("string"),
            "yyyyMMdd"
        )
    )
    .withColumn(
        "claim_month",
        F.date_format("claim_date", "yyyy-MM")
    )
)

claim_lines_with_month.select(
    "CLM_FROM_DT",
    "claim_date",
    "claim_month"
).show(10) 

+-----------+----------+-----------+
|CLM_FROM_DT|claim_date|claim_month|
+-----------+----------+-----------+
|   20091005|2009-10-05|    2009-10|
|   20091006|2009-10-06|    2009-10|
|   20091031|2009-10-31|    2009-10|
|   20091110|2009-11-10|    2009-11|
|   20091111|2009-11-11|    2009-11|
|   20091117|2009-11-17|    2009-11|
|   20091119|2009-11-19|    2009-11|
|   20091119|2009-11-19|    2009-11|
|   20091205|2009-12-05|    2009-12|
|   20091212|2009-12-12|    2009-12|
+-----------+----------+-----------+
only showing top 10 rows


In [18]:
#activity by provider and month
monthly_provider_activity = (
    claim_lines_with_month
    .groupBy("provider_npi", "claim_month")
    .agg(
        F.count("*").alias("monthly_claim_lines"),
        F.countDistinct("CLM_ID").alias("monthly_unique_claims"),
        F.countDistinct("DESYNPUF_ID").alias("monthly_unique_beneficiaries"),
        F.sum("payment_amount").alias("monthly_payment"),
        F.countDistinct("hcpcs_code").alias("monthly_unique_hcpcs")
    )
    .orderBy("provider_npi", "claim_month")
)

monthly_provider_activity.show(20, truncate=False)

+------------+-----------+-------------------+---------------------+----------------------------+---------------+--------------------+
|provider_npi|claim_month|monthly_claim_lines|monthly_unique_claims|monthly_unique_beneficiaries|monthly_payment|monthly_unique_hcpcs|
+------------+-----------+-------------------+---------------------+----------------------------+---------------+--------------------+
|NULL        |2008-01    |776                |362                  |354                         |22280.0        |246                 |
|NULL        |2008-02    |1023               |451                  |443                         |27260.0        |300                 |
|NULL        |2008-03    |1004               |497                  |485                         |30560.0        |316                 |
|NULL        |2008-04    |1090               |523                  |515                         |35180.0        |323                 |
|NULL        |2008-05    |1048               |474      

In [19]:
#missing provider Ids
provider_null_counts = (
    claim_lines_parquet
    .select(
        F.count("*").alias("total_lines"),
        F.sum(
            F.when(F.col("provider_npi").isNull(), 1).otherwise(0)
        ).alias("null_provider_lines")
    )
)

provider_null_counts.show()

provider_null_counts = provider_null_counts.withColumn(
    "null_provider_percentage",
    F.round(
        F.col("null_provider_lines") / F.col("total_lines") * 100,
        2
    )
)

provider_null_counts.show()

+-----------+-------------------+
|total_lines|null_provider_lines|
+-----------+-------------------+
|    4425332|               9421|
+-----------+-------------------+

+-----------+-------------------+------------------------+
|total_lines|null_provider_lines|null_provider_percentage|
+-----------+-------------------+------------------------+
|    4425332|               9421|                    0.21|
+-----------+-------------------+------------------------+



In [20]:
#only use identifiable providers
claim_lines_with_provider = (
    claim_lines_parquet
    .filter(F.col("provider_npi").isNotNull())
)

print("Claim lines with provider:", claim_lines_with_provider.count())

Claim lines with provider: 4415911


In [21]:
#Use filtered dataframe so null providers are removed
monthly_provider_activity = (
    claim_lines_with_provider
    .withColumn(
        "claim_date",
        F.to_date(
            F.col("CLM_FROM_DT").cast("string"),
            "yyyyMMdd"
        )
    )
    .withColumn(
        "claim_month",
        F.date_format("claim_date", "yyyy-MM")
    )
    .groupBy("provider_npi", "claim_month")
    .agg(
        F.count("*").alias("monthly_claim_lines"),
        F.countDistinct("CLM_ID").alias("monthly_unique_claims"),
        F.countDistinct("DESYNPUF_ID").alias("monthly_unique_beneficiaries"),
        F.sum("payment_amount").alias("monthly_payment"),
        F.countDistinct("hcpcs_code").alias("monthly_unique_hcpcs")
    )
    .orderBy("provider_npi", "claim_month")
)
monthly_provider_activity.show(20, truncate=False)

+------------+-----------+-------------------+---------------------+----------------------------+---------------+--------------------+
|provider_npi|claim_month|monthly_claim_lines|monthly_unique_claims|monthly_unique_beneficiaries|monthly_payment|monthly_unique_hcpcs|
+------------+-----------+-------------------+---------------------+----------------------------+---------------+--------------------+
|6594        |2008-08    |6                  |1                    |1                           |60.0           |5                   |
|6594        |2009-05    |2                  |1                    |1                           |60.0           |2                   |
|42589       |2009-01    |2                  |1                    |1                           |10.0           |2                   |
|54762       |2009-11    |2                  |1                    |1                           |100.0          |2                   |
|64648       |2008-07    |4                  |1        

In [22]:
provider_hcpcs_summary = (
    claim_lines_with_provider
    .groupBy("provider_npi", "hcpcs_code")
    .agg(
        F.count("*").alias("claim_lines"),
        F.countDistinct("CLM_ID").alias("unique_claims"),
        F.countDistinct("DESYNPUF_ID").alias("unique_beneficiaries"),
        F.sum("payment_amount").alias("total_payment")
    )
)

provider_hcpcs_summary.show(20, truncate=False)

provider_hcpcs_summary.write.mode("overwrite").parquet(
    "../data/processed/provider_hcpcs_summary"
)

print("Saved provider HCPCS summary.")

+------------+----------+-----------+-------------+--------------------+-------------+
|provider_npi|hcpcs_code|claim_lines|unique_claims|unique_beneficiaries|total_payment|
+------------+----------+-----------+-------------+--------------------+-------------+
|70593157    |00940     |1          |1            |1                   |60.0         |
|84070903    |99285     |1          |1            |1                   |140.0        |
|208049952   |98941     |1          |1            |1                   |50.0         |
|235073716   |11604     |1          |1            |1                   |310.0        |
|295463209   |72040     |1          |1            |1                   |20.0         |
|301654125   |36556     |1          |1            |1                   |0.0          |
|302412179   |93325     |1          |1            |1                   |0.0          |
|314210365   |G0008     |2          |1            |1                   |70.0         |
|321024610   |A0425     |1          |1     

Saved provider HCPCS summary.


In [23]:
#provider level time features (one row per provider, aggregated over time)
provider_time_features = (
    monthly_provider_activity
    .groupBy("provider_npi")
    .agg(
        F.countDistinct("claim_month").alias("active_months"),
        F.avg("monthly_claim_lines").alias("avg_monthly_claim_lines"),
        F.max("monthly_claim_lines").alias("max_monthly_claim_lines"),
        F.stddev("monthly_claim_lines").alias("std_monthly_claim_lines"),
        F.avg("monthly_payment").alias("avg_monthly_payment"),
        F.max("monthly_payment").alias("max_monthly_payment"),
        F.stddev("monthly_payment").alias("std_monthly_payment"),
        F.avg("monthly_unique_beneficiaries").alias(
            "avg_monthly_unique_beneficiaries"
        )
    )
)
provider_time_features.show(10, truncate=False)

+------------+-------------+-----------------------+-----------------------+-----------------------+-------------------+-------------------+-------------------+--------------------------------+
|provider_npi|active_months|avg_monthly_claim_lines|max_monthly_claim_lines|std_monthly_claim_lines|avg_monthly_payment|max_monthly_payment|std_monthly_payment|avg_monthly_unique_beneficiaries|
+------------+-------------+-----------------------+-----------------------+-----------------------+-------------------+-------------------+-------------------+--------------------------------+
|6357152     |1            |2.0                    |2                      |NULL                   |0.0                |0.0                |NULL               |1.0                             |
|9693229     |2            |1.0                    |1                      |0.0                    |35.0               |50.0               |21.213203435596427 |1.0                             |
|20428197    |8            |2.

In [24]:
#combine provider_features (overall provider behavior) and provider_time_features (monthly activity behavior)
provider_features_combined = (
    provider_features
    .join(
        provider_time_features,
        on="provider_npi",
        how="left"
    )
)

provider_features_combined.show(10, truncate=False)

print(
    "Number of providers:",
    provider_features_combined.select("provider_npi").distinct().count()
)

+------------+-----------------+-------------+--------------------+-------------+------------------+--------------------+------------------+------------+-------------+-----------------------+-----------------------+-----------------------+-------------------+-------------------+-------------------+--------------------------------+
|provider_npi|total_claim_lines|unique_claims|unique_beneficiaries|total_payment|avg_payment       |total_allowed_charge|avg_allowed_charge|unique_hcpcs|active_months|avg_monthly_claim_lines|max_monthly_claim_lines|std_monthly_claim_lines|avg_monthly_payment|max_monthly_payment|std_monthly_payment|avg_monthly_unique_beneficiaries|
+------------+-----------------+-------------+--------------------+-------------+------------------+--------------------+------------------+------------+-------------+-----------------------+-----------------------+-----------------------+-------------------+-------------------+-------------------+--------------------------------+
|

In [25]:
#feature distributions, which features are heavily skewed
provider_features_combined.select(
    "total_claim_lines",
    "unique_claims",
    "unique_beneficiaries",
    "total_payment",
    "avg_payment",
    "unique_hcpcs",
    "active_months",
    "avg_monthly_claim_lines",
    "max_monthly_claim_lines",
    "avg_monthly_payment"
).describe().show()

#Most providers are pretty small
#total paid has a huge range
#avg payment is more stable
#one provider is missing time-based feature

+-------+-----------------+------------------+--------------------+-----------------+-----------------+------------------+------------------+-----------------------+-----------------------+-------------------+
|summary|total_claim_lines|     unique_claims|unique_beneficiaries|    total_payment|      avg_payment|      unique_hcpcs|     active_months|avg_monthly_claim_lines|max_monthly_claim_lines|avg_monthly_payment|
+-------+-----------------+------------------+--------------------+-----------------+-----------------+------------------+------------------+-----------------------+-----------------------+-------------------+
|  count|           498095|            498095|              498095|           498095|           498095|            498095|            498094|                 498094|                 498094|             498094|
|   mean| 8.88451399833365| 4.740579608307653|    4.09309469077184|395.8981720354551|48.40020617088741| 6.767339563737841|3.5530120820568003|      2.01735509663

In [26]:
#log transformed features, highly skewed dollar and count features, using log(1+x) to avoid log(0) issues
#the log transformation compresses extreme values while preserving ordering. Prevents very large providers from dominating anomaly modeling. 
from pyspark.sql import functions as F

provider_features_model = (
    provider_features_combined
    .withColumn(
        "log_total_claim_lines",
        F.log1p("total_claim_lines")
    )
    .withColumn(
        "log_unique_claims",
        F.log1p("unique_claims")
    )
    .withColumn(
        "log_unique_beneficiaries",
        F.log1p("unique_beneficiaries")
    )
    .withColumn(
        "log_total_payment",
        F.log1p("total_payment")
    )
    .withColumn(
        "log_unique_hcpcs",
        F.log1p("unique_hcpcs")
    )
    .withColumn(
        "log_avg_monthly_claim_lines",
        F.log1p("avg_monthly_claim_lines")
    )
    .withColumn(
        "log_max_monthly_claim_lines",
        F.log1p("max_monthly_claim_lines")
    )
    .withColumn(
        "log_avg_monthly_payment",
        F.log1p("avg_monthly_payment")
    )
)

provider_features_model.select(
    "total_claim_lines",
    "log_total_claim_lines",
    "total_payment",
    "log_total_payment",
    "avg_monthly_payment",
    "log_avg_monthly_payment"
).show(10, truncate=False)

+-----------------+---------------------+-------------+------------------+-------------------+-----------------------+
|total_claim_lines|log_total_claim_lines|total_payment|log_total_payment |avg_monthly_payment|log_avg_monthly_payment|
+-----------------+---------------------+-------------+------------------+-------------------+-----------------------+
|4                |1.6094379124341003   |240.0        |5.484796933490655 |120.0              |4.795790545596741      |
|16               |2.833213344056216    |830.0        |6.7226297948554485|166.0              |5.117993812416755      |
|9                |2.302585092994046    |360.0        |5.8888779583328805|72.0               |4.290459441148391      |
|4                |1.6094379124341003   |100.0        |4.61512051684126  |33.333333333333336 |3.5361166995615263     |
|21               |3.091042453358316    |350.0        |5.860786223465865 |31.818181818181817 |3.4909826855345103     |
|3                |1.3862943611198906   |10.0   

In [27]:
#Selecting modeling features
#Mix of: overall volume, payment behavior, service types, activity, variability

#feature dataframe using log transformed variables
model_features = provider_features_model.select(
    "provider_npi",

    # Overall activity
    "log_total_claim_lines", #claim volume
    "log_unique_claims", #distinct claims
    "log_unique_beneficiaries", #patients
    "log_unique_hcpcs", #distinct procedures

    # Payment behavior
    "log_total_payment", #total paid
    "avg_payment", #average paid per claim

    # Time-based activity
    "active_months",
    "log_avg_monthly_claim_lines",
    "log_max_monthly_claim_lines",
    "log_avg_monthly_payment",

    # Variability
    "std_monthly_claim_lines", #standard deviation of monthly claim volume
    "std_monthly_payment" #standard deviation of monthly payment
)

model_features.show(10, truncate=False)

+------------+---------------------+------------------+------------------------+------------------+------------------+------------------+-------------+---------------------------+---------------------------+-----------------------+-----------------------+-------------------+
|provider_npi|log_total_claim_lines|log_unique_claims |log_unique_beneficiaries|log_unique_hcpcs  |log_total_payment |avg_payment       |active_months|log_avg_monthly_claim_lines|log_max_monthly_claim_lines|log_avg_monthly_payment|std_monthly_claim_lines|std_monthly_payment|
+------------+---------------------+------------------+------------------------+------------------+------------------+------------------+-------------+---------------------------+---------------------------+-----------------------+-----------------------+-------------------+
|27409610    |3.4965075614664802   |3.1354942159291497|3.044522437723423       |3.258096538021482 |7.58629630715272  |61.5625           |14           |1.1895840668738362   

In [28]:
#check missing values
from pyspark.sql import functions as F

model_features.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in model_features.columns
]).show()

#lots of missing columns in std_monthly_claim_lines and std_monthly_payment, likely because these providers only have one month of activity.

+------------+---------------------+-----------------+------------------------+----------------+-----------------+-----------+-------------+---------------------------+---------------------------+-----------------------+-----------------------+-------------------+
|provider_npi|log_total_claim_lines|log_unique_claims|log_unique_beneficiaries|log_unique_hcpcs|log_total_payment|avg_payment|active_months|log_avg_monthly_claim_lines|log_max_monthly_claim_lines|log_avg_monthly_payment|std_monthly_claim_lines|std_monthly_payment|
+------------+---------------------+-----------------+------------------------+----------------+-----------------+-----------+-------------+---------------------------+---------------------------+-----------------------+-----------------------+-------------------+
|           1|                    0|                0|                       0|               0|                0|          0|            1|                          1|                          1|         

In [29]:
# Standard deviation is undefined for providers with only one active month,
# so treat the lack of variation as 0.
model_features = model_features.fillna({
    "std_monthly_claim_lines": 0.0,
    "std_monthly_payment": 0.0
})

#check remaining null values
model_features.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in model_features.columns
]).show()

#look at the one provider with null values
model_features.filter(
    F.col("active_months").isNull()
).show(truncate=False)

#check original combined table
provider_features_combined.filter(
    F.col("provider_npi").isNull()
).show(truncate=False)

+------------+---------------------+-----------------+------------------------+----------------+-----------------+-----------+-------------+---------------------------+---------------------------+-----------------------+-----------------------+-------------------+
|provider_npi|log_total_claim_lines|log_unique_claims|log_unique_beneficiaries|log_unique_hcpcs|log_total_payment|avg_payment|active_months|log_avg_monthly_claim_lines|log_max_monthly_claim_lines|log_avg_monthly_payment|std_monthly_claim_lines|std_monthly_payment|
+------------+---------------------+-----------------+------------------------+----------------+-----------------+-----------+-------------+---------------------------+---------------------------+-----------------------+-----------------------+-------------------+
|           1|                    0|                0|                       0|               0|                0|          0|            1|                          1|                          1|         

+------------+---------------------+-----------------+------------------------+----------------+------------------+-----------------+-------------+---------------------------+---------------------------+-----------------------+-----------------------+-------------------+
|provider_npi|log_total_claim_lines|log_unique_claims|log_unique_beneficiaries|log_unique_hcpcs|log_total_payment |avg_payment      |active_months|log_avg_monthly_claim_lines|log_max_monthly_claim_lines|log_avg_monthly_payment|std_monthly_claim_lines|std_monthly_payment|
+------------+---------------------+-----------------+------------------------+----------------+------------------+-----------------+-------------+---------------------------+---------------------------+-----------------------+-----------------------+-------------------+
|NULL        |9.150802659259986    |8.424200324567073|8.317277766221235       |6.93828448401696|12.653769953351677|33.21728054346672|NULL         |NULL                       |NULL     

In [30]:
#Filter out null npi provider row
model_features = model_features.filter(
    F.col("provider_npi").isNotNull()
)

# Verify no missing values remain
model_features.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in model_features.columns
]).show()

# Verify provider count
print(
    "Number of providers:",
    model_features.select("provider_npi").distinct().count()
)

+------------+---------------------+-----------------+------------------------+----------------+-----------------+-----------+-------------+---------------------------+---------------------------+-----------------------+-----------------------+-------------------+
|provider_npi|log_total_claim_lines|log_unique_claims|log_unique_beneficiaries|log_unique_hcpcs|log_total_payment|avg_payment|active_months|log_avg_monthly_claim_lines|log_max_monthly_claim_lines|log_avg_monthly_payment|std_monthly_claim_lines|std_monthly_payment|
+------------+---------------------+-----------------+------------------------+----------------+-----------------+-----------+-------------+---------------------------+---------------------------+-----------------------+-----------------------+-------------------+
|           0|                    0|                0|                       0|               0|                0|          0|            0|                          0|                          0|         

In [31]:
#feature list
feature_cols = [
    "log_total_claim_lines",
    "log_unique_claims",
    "log_unique_beneficiaries",
    "log_unique_hcpcs",
    "log_total_payment",
    "avg_payment",
    "active_months",
    "log_avg_monthly_claim_lines",
    "log_max_monthly_claim_lines",
    "log_avg_monthly_payment",
    "std_monthly_claim_lines",
    "std_monthly_payment"
]

#convert to python
model_pd = model_features.select(
    "provider_npi",
    *feature_cols
).toPandas()

model_pd.head()

#check shape
print("Rows:", len(model_pd))
print("Features:", len(feature_cols))

#498904 providers, 12 features converted to pandas

Rows: 498094
Features: 12


In [32]:
#scaling features, some features r on dif scales
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X = scaler.fit_transform(
    model_pd[feature_cols]
)

print("X shape:", X.shape)

X shape: (498094, 12)


In [33]:
#fit isolation forest model- unsupervised ML model used for anomaly detection
#start by assuming 1% of providers are fraudulent, can adjust later
from sklearn.ensemble import IsolationForest

iso_forest = IsolationForest(
    n_estimators=200, #builds 200 isolation trees, more trees = better performance but slower
    contamination=0.01, #flags most unusual 1% of providers
    random_state=42, #for reproducibility
    n_jobs=-1 #use all available CPU cores for parallel processing
)

iso_forest.fit(X)

print("Model fitted successfully.")

Model fitted successfully.


In [34]:
#generate anomaly scores 
model_pd["anomaly_prediction"] = iso_forest.predict(X)
model_pd["anomaly_score"] = iso_forest.decision_function(X)
#results
model_pd.head()

#flagged providers:
print(
    model_pd["anomaly_prediction"].value_counts()
)

#inspect anomoalous providers
model_pd.sort_values(
    "anomaly_score",
    ascending=True
).head(20)

anomaly_prediction
 1    493113
-1      4981
Name: count, dtype: int64


,provider_npi,log_total_claim_lines,log_unique_claims,log_unique_beneficiaries,log_unique_hcpcs,log_total_payment,avg_payment,active_months,log_avg_monthly_claim_lines,log_max_monthly_claim_lines,log_avg_monthly_payment,std_monthly_claim_lines,std_monthly_payment,anomaly_prediction,anomaly_score
497551,9979265126,9.809342,9.187993,8.796188,7.361375,13.614228,44.922536,36,6.227744,6.552508,10.030751,151.544482,7156.817329,-1,-0.170108
177303,7132269719,9.179366,8.544614,7.661998,7.074963,13.011076,46.146070,36,5.599450,5.905362,9.427636,82.570701,4117.732578,-1,-0.170108
297860,1973668724,9.675080,9.040382,7.763021,7.297091,13.475343,44.715677,36,6.093758,6.456770,9.891873,130.859876,6348.605165,-1,-0.170108
197996,7961400834,9.453365,8.826294,8.535819,7.188413,13.275292,45.695686,36,5.872587,6.173786,9.691833,99.072412,4785.740891,-1,-0.170108
374593,5040521237,9.037890,8.390268,7.629004,7.005789,12.825334,44.148544,36,5.458521,5.805135,9.241909,77.854581,3231.932183,-1,-0.170108
413117,6585316279,9.201501,8.549273,8.278936,7.057898,12.998977,44.592877,36,5.621507,5.966147,9.415537,83.894771,4152.720279,-1,-0.170108
337345,3552796390,9.320181,8.698347,7.826842,7.158514,13.114434,44.448925,36,5.739793,6.049733,9.530986,85.947160,4448.033514,-1,-0.170108
22139,888901330,10.227417,9.611797,9.076695,7.491088,14.022192,44.469801,36,6.645163,6.918695,10.438702,221.662284,10011.419904,-1,-0.170108
272288,941734431,9.338470,8.716044,8.388223,7.100027,13.156380,45.512933,36,5.758025,6.104793,9.572929,96.838779,4952.326405,-1,-0.170108
257072,329757128,9.279307,8.657303,8.346168,7.075809,13.065287,44.082890,36,5.699049,6.089045,9.481842,94.419240,4562.760130,-1,-0.170108


In [35]:
#compare feature distributions for normal vs anomalous providers
model_pd.groupby("anomaly_prediction")[feature_cols].mean().T

#median comparison is often more informative for skewed data
model_pd.groupby("anomaly_prediction")[feature_cols].median().T

#results: model is finding providers that look unusual because they have much higher and variable activity

anomaly_prediction,-1,1
log_total_claim_lines,4.189655,1.609438
log_unique_claims,3.526361,1.098612
log_unique_beneficiaries,3.258097,1.098612
log_unique_hcpcs,3.891820,1.609438
log_total_payment,8.023225,5.017280
avg_payment,46.470078,37.500000
active_months,21.000000,2.000000
log_avg_monthly_claim_lines,1.510592,0.998529
log_max_monthly_claim_lines,2.564949,1.098612
log_avg_monthly_payment,5.129899,4.189655


In [36]:
import os

os.makedirs("../data/processed", exist_ok=True)

model_pd.to_parquet(
    "../data/processed/provider_anomaly_results.parquet",
    index=False
)

print("Saved anomaly results.")

import os
print(os.getcwd())

Saved anomaly results.
/Users/manyasood/Documents/healthcare-investigation/notebooks


In [37]:
#load monthly data
provider_time_features.write.mode("overwrite").parquet(
    "../data/processed/provider_time_features"
)

print("Saved monthly provider features.")

Saved monthly provider features.


In [38]:
[x for x in dir() if "month" in x.lower() or "time" in x.lower()]
[x for x in dir() if "claim" in x.lower() or "provider" in x.lower()]

['claim_line',
 'claim_line_dfs',
 'claim_lines',
 'claim_lines_1',
 'claim_lines_parquet',
 'claim_lines_with_month',
 'claim_lines_with_provider',
 'monthly_provider_activity',
 'provider_features',
 'provider_features_combined',
 'provider_features_model',
 'provider_hcpcs_summary',
 'provider_null_counts',
 'provider_time_features']

In [39]:
#to print provider monthly activity table
monthly_provider_activity.printSchema()
monthly_provider_activity.show(10)
monthly_provider_activity.columns

root
 |-- provider_npi: long (nullable = true)
 |-- claim_month: string (nullable = true)
 |-- monthly_claim_lines: long (nullable = false)
 |-- monthly_unique_claims: long (nullable = false)
 |-- monthly_unique_beneficiaries: long (nullable = false)
 |-- monthly_payment: double (nullable = true)
 |-- monthly_unique_hcpcs: long (nullable = false)



+------------+-----------+-------------------+---------------------+----------------------------+---------------+--------------------+
|provider_npi|claim_month|monthly_claim_lines|monthly_unique_claims|monthly_unique_beneficiaries|monthly_payment|monthly_unique_hcpcs|
+------------+-----------+-------------------+---------------------+----------------------------+---------------+--------------------+
|        6594|    2008-08|                  6|                    1|                           1|           60.0|                   5|
|        6594|    2009-05|                  2|                    1|                           1|           60.0|                   2|
|       42589|    2009-01|                  2|                    1|                           1|           10.0|                   2|
|       54762|    2009-11|                  2|                    1|                           1|          100.0|                   2|
|       64648|    2008-07|                  4|         

['provider_npi',
 'claim_month',
 'monthly_claim_lines',
 'monthly_unique_claims',
 'monthly_unique_beneficiaries',
 'monthly_payment',
 'monthly_unique_hcpcs']

In [40]:
import os

os.makedirs("../data/processed", exist_ok=True)

monthly_provider_activity.write.mode("overwrite").parquet(
    "../data/processed/monthly_provider_activity"
)

print("Saved monthly provider activity.")

Saved monthly provider activity.


In [ ]:
import os

print(
    os.path.exists("../data/processed/monthly_provider_activity")
)

True


26/10/05 23:42:49 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 822630 ms exceeds timeout 120000 ms
26/10/05 23:42:49 WARN SparkContext: Killing executors is not supported by current scheduler.
26/10/05 23:42:57 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:53)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:342)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:132)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$$